# XANES Spectroscopy Emulation for Battery Materials

This tutorial demonstrates emulating **X-ray Absorption Near Edge Structure (XANES)** 
spectra of lithium transition metal oxides (Li$_x$MO$_2$), a key characterization 
technique for battery research.

## Physical Context

XANES at the transition metal K-edge probes the local electronic structure:
- **Pre-edge peak**: 1s → 3d quadrupole transitions (sensitive to local symmetry)
- **Main edge**: 1s → 4p dipole transitions (sensitive to oxidation state)
- **White line**: Shape resonance (sensitive to coordination environment)

As lithium is extracted from cathode materials:
- Oxidation state increases (edge shifts to higher energy)
- Local symmetry distorts (pre-edge intensity changes)
- Jahn-Teller effects appear at certain compositions


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from smlr import Surrogate, StrengthDataset, StrengthSample
from smlr.metrics import normalized_l2


## Synthetic XANES Model

We model the K-edge XANES of a transition metal oxide with parameters:
- **x** (0-1): Lithium content (Li$_x$MO$_2$) - affects oxidation state
- **δ** (0-1): Jahn-Teller distortion parameter - affects local symmetry

The spectrum consists of:
1. **Pre-edge**: Small peak ~5 eV below main edge
2. **Main edge**: Arctangent step function (edge jump)
3. **White line**: Lorentzian peak at the edge
4. **EXAFS oscillations**: Damped oscillatory structure above the edge


In [ ]:
def xanes_spectrum(params, energy):
    """Generate synthetic XANES spectrum.
    
    Parameters
    ----------
    params : array-like
        [0]: x = lithium content (0=fully delithiated, 1=fully lithiated)
        [1]: delta = Jahn-Teller distortion (0=none, 1=strong)
    energy : array
        Energy axis relative to nominal edge (eV)
    
    Physical effects modeled:
    - Edge position shifts ~2 eV per unit change in oxidation state
    - Pre-edge intensity increases with distortion (broken centrosymmetry)
    - White line narrows and intensifies with delithiation
    """
    x, delta = params
    
    # Reference edge at 0 eV, shifts with oxidation state
    # Delithiation (lower x) → higher oxidation → edge shifts UP
    edge_shift = 2.0 * (1 - x)
    
    # Pre-edge: 1s→3d transition, forbidden in octahedral symmetry
    # Intensity increases with distortion (Jahn-Teller, 4d-2p mixing)
    pre_edge_pos = -5.0 + edge_shift
    pre_edge_intensity = 0.02 + 0.08 * delta + 0.03 * (1 - x)
    pre_edge_width = 1.0
    
    # Main edge: arctangent step
    edge_position = 0.0 + edge_shift
    edge_height = 1.0
    edge_width = 1.5 - 0.3 * (1 - x)  # Sharper for higher oxidation states
    
    # White line: resonance peak at edge
    wl_position = 3.0 + edge_shift
    wl_intensity = 0.5 + 0.3 * (1 - x)  # Stronger for delithiated
    wl_width = 2.5 - 0.8 * (1 - x)  # Narrower for delithiated
    
    # Second feature (multiple scattering)
    ms_position = 12.0 + edge_shift
    ms_intensity = 0.15
    ms_width = 4.0
    
    # Component functions
    def lorentzian(e, e0, gamma, amp):
        return amp * gamma**2 / ((e - e0)**2 + gamma**2)
    
    def edge_step(e, e0, width, height):
        return height * (0.5 + np.arctan((e - e0) / width) / np.pi)
    
    def exafs_oscillation(e, e0, amp, period, decay):
        """Simplified EXAFS-like oscillations above the edge."""
        k = np.sqrt(np.maximum(e - e0, 0))  # Convert to k-space
        return amp * np.sin(period * k) * np.exp(-decay * k) * (e > e0)
    
    # Combine all components
    spectrum = (
        edge_step(energy, edge_position, edge_width, edge_height) +
        lorentzian(energy, pre_edge_pos, pre_edge_width, pre_edge_intensity) +
        lorentzian(energy, wl_position, wl_width, wl_intensity) +
        lorentzian(energy, ms_position, ms_width, ms_intensity) +
        exafs_oscillation(energy, edge_position + 5, 0.05, 2.0, 0.15)
    )
    
    return spectrum

# Create training data across composition-distortion space
energy = np.linspace(-15, 50, 300)  # Relative to nominal edge

# Dense sampling near phase transitions
x_vals = np.array([0.0, 0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 0.9, 1.0])
delta_vals = np.linspace(0, 1, 6)

samples = []
for x in x_vals:
    for delta in delta_vals:
        params = np.array([x, delta])
        spectrum = xanes_spectrum(params, energy)
        samples.append(StrengthSample(params, energy, spectrum))

dataset = StrengthDataset(samples)
print(f"Created {len(samples)} training spectra")
print(f"Parameter ranges: x ∈ [0, 1], δ ∈ [0, 1]")


## Visualize Training Data

Let's examine how the spectra change with lithium content and distortion.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel 1: Effect of lithium content (no distortion)
ax = axes[0]
colors = plt.cm.viridis(np.linspace(0, 1, len(x_vals)))
for i, x in enumerate(x_vals):
    for sample in dataset.samples:
        if np.isclose(sample.params[0], x) and np.isclose(sample.params[1], 0.0):
            ax.plot(sample.energy, sample.strength, color=colors[i], 
                   label=f"x={x:.1f}")
            break
ax.set_xlabel("Energy relative to edge (eV)")
ax.set_ylabel("Absorption (a.u.)")
ax.set_title("Effect of Li content (δ=0)")
ax.set_xlim(-10, 40)
ax.legend(loc='upper right', fontsize=8)

# Panel 2: Effect of Jahn-Teller distortion (x=0.5)
ax = axes[1]
colors = plt.cm.plasma(np.linspace(0, 1, len(delta_vals)))
for i, delta in enumerate(delta_vals):
    for sample in dataset.samples:
        if np.isclose(sample.params[0], 0.5) and np.isclose(sample.params[1], delta):
            ax.plot(sample.energy, sample.strength, color=colors[i],
                   label=f"δ={delta:.1f}")
            break
ax.set_xlabel("Energy relative to edge (eV)")
ax.set_ylabel("Absorption (a.u.)")
ax.set_title("Effect of JT distortion (x=0.5)")
ax.set_xlim(-10, 40)
ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
fig


## Train PMM Surrogate

The Parametric Matrix Model captures the physics of the shifting and 
reshaping spectral features as the material properties change.

In [ ]:
# Train PMM surrogate
model = Surrogate('pmm', n_poles=15, retain=0.8)
model.fit(dataset)
print("Model trained successfully")

# Test on interpolation point (not in training set)
test_params = np.array([0.33, 0.4])  # Intermediate composition and distortion
result = model.predict(test_params, energy)

# Compare to ground truth
truth = xanes_spectrum(test_params, energy)
error = normalized_l2(result.spectrum, truth, energy)

print(f"Test point: x={test_params[0]:.2f}, δ={test_params[1]:.2f}")
print(f"Normalized L2 error: {error:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: Full spectrum comparison
ax = axes[0]
ax.plot(energy, truth, 'k-', lw=2, label='Ground truth')
ax.plot(energy, result.spectrum, 'r--', lw=2, alpha=0.8, label='PMM prediction')
ax.set_xlabel("Energy relative to edge (eV)")
ax.set_ylabel("Absorption (a.u.)")
ax.set_title(f"XANES Emulation (x={test_params[0]:.2f}, δ={test_params[1]:.2f})")
ax.set_xlim(-10, 40)
ax.legend()

# Right: Pre-edge detail
ax = axes[1]
mask = (energy > -10) & (energy < 5)
ax.plot(energy[mask], truth[mask], 'k-', lw=2, label='Ground truth')
ax.plot(energy[mask], result.spectrum[mask], 'r--', lw=2, alpha=0.8, label='PMM prediction')
ax.set_xlabel("Energy relative to edge (eV)")
ax.set_ylabel("Absorption (a.u.)")
ax.set_title("Pre-edge Region Detail")
ax.legend()

plt.tight_layout()
fig


## Application: Rapid Screening

With a trained surrogate, we can rapidly predict XANES across the entire
parameter space - useful for:
- **Operando analysis**: Real-time composition tracking during battery cycling
- **High-throughput screening**: Evaluating candidate materials before synthesis
- **Inverse problems**: Fitting experimental spectra to extract composition

In [ ]:
# Predict across composition space
x_test = np.linspace(0, 1, 20)
predictions = []

for x in x_test:
    test_p = np.array([x, 0.3])  # Fixed distortion
    result = model.predict(test_p, energy)
    predictions.append(result.spectrum)

predictions = np.array(predictions)

# Create a spectral map
fig, ax = plt.subplots(figsize=(10, 5))
mesh = ax.pcolormesh(energy, x_test, predictions, shading='auto', cmap='viridis')
ax.set_xlabel("Energy relative to edge (eV)")
ax.set_ylabel("Li content x")
ax.set_title("Predicted XANES evolution with lithiation state")
ax.set_xlim(-10, 40)
plt.colorbar(mesh, ax=ax, label="Absorption")
fig


## Summary

This example demonstrated:
1. **Realistic spectral physics**: Edge shifts, pre-edge intensity, white line changes
2. **Efficient emulation**: PMM captures complex energy-dependent transformations
3. **Rapid prediction**: Sub-millisecond predictions enable real-time analysis

The trained surrogate can replace expensive *ab initio* XANES calculations
(e.g., FEFF, OCEAN) for interactive analysis and high-throughput screening.
